# Leesmateriaal recommender

Deze notebook brengt de analyse, tekstbewerking en aanbevelingsalgoritmen samen. De codecellen zijn overgenomen uit de bestaande notebooks; de extra markdown licht de onderdelen en hun volgorde toe.



## 1. Analyse

In dit hoofdstuk worden de brongegevens verkend en opgeschoond voordat ze voor de aanbevelingen worden gebruikt.



### Leesmateriaal

We controleren en normaliseren de catalogusgegevens, bekijken datakwaliteit en onderzoeken de numerieke kenmerken van het leesmateriaal.



In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("./dataset/LeesmateriaalCSV.csv", sep=";")

In [ ]:
# Controleren of Themas_boek en trefwoorden per rij identiek zijn
df["Themas_boek"] = (
    df["Themas_boek"]
    .astype("string")
    .fillna("")
    .str.replace(r"(?:https?://|www\.)\S+(?:\s+\[[^\]]+\])?", "", regex=True, case=False)
    .str.replace(r"\[[^\]]+\]", "", regex=True)
    .str.strip(" ,;")
)
themas_boek = df["Themas_boek"].astype("string").fillna("").map(
    lambda waarde: ";".join(woord.strip() for woord in waarde.split(";"))
)
trefwoorden = df["trefwoorden"].astype("string").fillna("").map(
    lambda waarde: ";".join(woord.strip() for woord in waarde.split(";"))
)
zelfde = themas_boek.eq(trefwoorden)
verschillen = df.loc[~zelfde, ["Titel", "Themas_boek", "trefwoorden"]]

print(f"Alle waarden identiek: {verschillen.empty}")
verschillen

In [ ]:
# Controleren of elk onderwerp in de trefwoorden voorkomt
onderwerpen = df["onderwerp"].astype("string").fillna("").str.strip().str.casefold()
trefwoorden = [
    [woord.strip().casefold() for woord in rij.split(";") if woord.strip()]
    for rij in df["trefwoorden"].astype("string").fillna("")
]
onderwerp_gevonden = pd.Series(
    [
        bool(onderwerp) and onderwerp in woorden
        for onderwerp, woorden in zip(onderwerpen, trefwoorden)
    ],
    index=df.index,
)
ontbreekt = df.loc[
    onderwerpen.ne("") & ~onderwerp_gevonden,
    ["Titel", "onderwerp", "trefwoorden"],
]
print(f"Alle onderwerpen staan in trefwoorden: {ontbreekt.empty}")
ontbreekt

In [ ]:
# Controleren of ingevulde genres_kopie al in genre staan
genre = df["genre"].astype("string").fillna("").str.strip().str.casefold()
genre_kopie = df["genre_kopie"].astype("string").fillna("").str.strip().str.casefold()
extra_genres = genre_kopie.ne("") & genre_kopie.ne(genre)
verschillen_genre = df.loc[extra_genres, ["Titel", "genre", "genre_kopie"]]

print(f"Alle ingevulde genres_kopie komen overeen met genre: {verschillen_genre.empty}")
verschillen_genre

In [ ]:
#onnodige kolommen verwijderen
df = df.drop(columns=['Unnamed: 3', 'Themas_boek', 'materiaal_id', 'interne_batchcode', 'genre_kopie', 'woordenaantal_omschrijving', 'aandeel_lange_woorden_pct', 'aantal_zinnen', 'administratieve_prioriteit'])

In [ ]:
# Genre uit Type materiaal verplaatsen als genre leeg is
mask = df["Type materiaal"].str.match(r"(?i)^boek\s*-", na=False)
genre = df.loc[mask, "Type materiaal"].str.split("-", n=1).str[1].str.strip()
missing_genre = df["genre"].isna()
df.loc[mask & missing_genre, "genre"] = genre.loc[mask & missing_genre]

In [ ]:
#Type materiaal normaliseren
mask = df["Type materiaal"].str.match(r"(?i)^boek\s*-", na=False)
df.loc[mask, "Type materiaal"] = "boek"
df.loc[df["Type materiaal"] == "Boek", "Type materiaal"] = "boek"
df.loc[df["Type materiaal"] == "digitale krant", "Type materiaal"] = "krant (digitaal)"

In [ ]:
#Taal normaliseren
df["taal"] = "NL"

In [ ]:
#Doelgroep normaliseren
df.loc[df["doelgroep"] == "mbo", "doelgroep"] = "MBO"
df.loc[df["doelgroep"] == "mbo / nt2", "doelgroep"] = "MBO/NT2"

In [ ]:
#Doelgroepen combineren
def combineer_doelgroepen(row):
    groepen = []

    for waarde in row.dropna():
        for groep in str(waarde).split("/"):
            groep = groep.strip()

            if groep != "" and groep not in groepen:
                groepen.append(groep)

    volgorde = {"MBO": 0, "NT2": 1}
    groepen.sort(key=lambda groep: volgorde.get(groep, 2))

    return "/".join(groepen)

df["doelgroep"] = df[["doelgroep", "doelgroep_extra"]].apply(
    combineer_doelgroepen, axis=1
)
df.loc[df["doelgroep"] == "", "doelgroep"] = np.nan

df = df.drop(["doelgroep_extra"], axis=1)

In [ ]:
#onrealistische waarden in gem_woordlengte vervangen door NaN
df.loc[df["gem_woordlengte"] > 10, ["gem_woordlengte"]] = np.nan

In [ ]:
#interessecluster normaliseren
df["interessecluster"] = df["interessecluster"].str.lower()

In [ ]:
#onrealistische en onbekende jaartallen in publicatiejaar vervangen door 'onbekend'
df.loc[df["publicatiejaar"] == 'onbekend', ["publicatiejaar"]] = np.nan
df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")
df.loc[df["publicatiejaar"] > 2026, ["publicatiejaar"]] = np.nan

In [ ]:
#beschikbaar_formaat normaliseren
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.lower()

In [ ]:
#aantal_paginas normaliseren
df.loc[df["aantal_paginas"] < 1, "aantal_paginas"] = np.nan
df["aantal_paginas"] = df["aantal_paginas"].astype('Int64')

In [ ]:
#geschatte_leestijd_min normaliseren
df.loc[df["geschatte_leestijd_min"] < 1, "geschatte_leestijd_min"] = np.nan

In [ ]:
#Gegevensstatus opschonen en veranderen naar bruikbare data
df.loc[df["gegevensstatus"] == "Aangevuld (deels gesimuleerd)", ["gegevensstatus"]] = 'Beschikbaar'
df.loc[df["gegevensstatus"] == "gecontroleerd", ["gegevensstatus"]] = 'Binnenkort Terug Verwacht'
df.loc[df["gegevensstatus"] == "In behandeling", ["gegevensstatus"]] = 'Uitgeleend'
df.loc[df["gegevensstatus"].isna(), ["gegevensstatus"]] = 'Onbekend'

In [ ]:
# Korte url-vermelding uit bron_url verwijderen
df["bron_url"] = df["bron_url"].str.replace(
    r"((?:https?://|www\.)\S+?)(?:\s*,?\s*\[[^\]]+\])+",
    r"\1",
    regex=True,
)

In [ ]:
# Auteurs scheiden met een slash
df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s+en\s+", ";", case=False, regex=True)
    .str.replace(r"\s*[,&]\s*", ";", regex=True)
    .str.replace(r"\s*/\s*", ";", regex=True)
    .str.strip()
)

In [ ]:
df["trefwoorden"] = df["trefwoorden"].str.replace("; ", ";")

In [ ]:
ag = df[df["aantal_paginas"].notna() & df["geschatte_leestijd_min"].notna()][["aantal_paginas", "geschatte_leestijd_min"]]

,aantal_paginas,geschatte_leestijd_min
0,265.0,556.0
1,171.0,184.0
2,219.0,662.0
3,288.0,748.0
4,358.0,196.0


In [ ]:
df[df["aantal_paginas"].isna() | df["geschatte_leestijd_min"].isna()]

In [ ]:
gem_paginas_p_min = ag["aantal_paginas"].mean() / ag["geschatte_leestijd_min"].mean()
gem_min_p_p = ag["geschatte_leestijd_min"].mean() / ag["aantal_paginas"].mean()

df.loc[df["aantal_paginas"].isna() & df["geschatte_leestijd_min"].notna(), "aantal_paginas"] = (df["geschatte_leestijd_min"] * gem_paginas_p_min).round().astype("Int64")
df.loc[df["geschatte_leestijd_min"].isna() & df["aantal_paginas"].notna(), "geschatte_leestijd_min"] = df["aantal_paginas"] * gem_min_p_p

In [ ]:
def create_new_row(df):
    kolommen = ["Korte omschrijving", "genre", "onderwerp", "trefwoorden", "Niveau", "uitgever", "beschikbaar_formaat", "Auteur"]
    delen = df[kolommen].fillna("").astype("string")
    df["Gecombineerd"] = delen.apply(
        lambda rij: "/".join(deel for deel in rij if deel.strip()),
        axis=1,
    )
    return df

In [ ]:
df = create_new_row(df)

In [ ]:
df.to_csv("./dataset/cleaned_Leesmateriaal.csv", index=False)

In [ ]:
df.head()

In [ ]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

#### Dimensionality reduction op de opgeschoonde data (PCA, t-SNE, UMAP)

**Wat gaan we doen?**
Elk boek heeft 5 cijfers (pagina's, leestijd, woordlengte, zinslengte en leescomplexiteit). Dat zijn te veel cijfers om in één plaatje te zetten. Met PCA, t-SNE en UMAP maken we er een plaatje van met maar 2 richtingen (links/rechts en omhoog/omlaag). Boeken die op elkaar lijken komen dan dicht bij elkaar te staan. Zo zien we of boeken van hetzelfde niveau op elkaar lijken.

**Wat gebeurt er in de code hieronder?**
1. **Features kiezen:** we kiezen de 5 getal-kolommen die we gebruiken om boeken te vergelijken.
2. **Label kiezen:** we kiezen `Niveau` (2F, 3F, 3F+) om de punten later mee te kleuren. Het label gebruiken de technieken zelf niet, het is alleen voor ons om te kijken of boeken van hetzelfde niveau bij elkaar komen te liggen.
3. **Kopie maken zonder lege waardes:** de technieken kunnen niet tegen lege vakjes. Daarom maken we een kopie (`df_analyse`) met alleen de boeken waar alle 5 de cijfers zijn ingevuld. De originele tabel `df` blijft zoals hij was, zodat we de lege waardes later nog kunnen invullen.
4. **Schalen:** leestijd is een groot getal (honderden) en woordlengte een klein getal (rond 6). Zonder schalen telt het grote getal te zwaar mee. Met `StandardScaler` brengen we alle kolommen op dezelfde schaal.
5. **Plotfunctie:** `plot_2d` is een hulpfunctie die een plaatje maakt waarin elk punt een boek is, gekleurd per niveau. Die gebruiken we straks bij alle drie de technieken.

De code print ook hoeveel boeken er meedoen en hoeveel er zijn weggelaten door lege waardes.

In [ ]:
# --- Features, labels en schalen ---
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1) Welke numerieke kolommen gebruiken we als features, en waarop kleuren we de punten?
features = [
    "aantal_paginas",
    "geschatte_leestijd_min",
    "gem_woordlengte",
    "gem_zinslengte_woorden",
    "leescomplexiteit_score",
]
label_kolom = "Niveau"

# 2) Kopie met alleen volledige rijen (df zelf blijft ongemoeid)
# LET OP: na feature engineering vervang je hier df door de ingevulde dataframe
# en laat je (indien alles is ingevuld) .dropna() alleen nog voor het label staan.
df_analyse = df[features + [label_kolom]].dropna().copy()
df_analyse[features] = df_analyse[features].astype(float)  # Int64 -> float

X = df_analyse[features].to_numpy()
y = df_analyse[label_kolom].to_numpy()   # labels (2F, 2F-3F, 3F, 3F+) om punten te kleuren

# 3) Schalen (gemiddelde 0, std 1): leestijd loopt in honderden, woordlengte rond 6
X_scaled = StandardScaler().fit_transform(X)

print(f"Rijen in df: {len(df)} | rijen in analyse: {len(df_analyse)} "
      f"({len(df) - len(df_analyse)} weggelaten door lege waardes)")
print("X_scaled shape:", X_scaled.shape)
print(df_analyse[label_kolom].value_counts())


def plot_2d(X_2d, titel, xlabel, ylabel):
    """Scatterplot per label (Niveau) met legenda."""
    plt.figure(figsize=(7, 5))
    for label in sorted(np.unique(y)):
        m = y == label
        plt.scatter(X_2d[m, 0], X_2d[m, 1], s=40, alpha=0.8, label=label)
    plt.title(titel)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(title=label_kolom)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.show()

##### PCA

**Wat doet PCA?**
PCA zoekt de "beste hoek" om naar je boeken te kijken. Het maakt van de 5 cijfers per boek 2 nieuwe cijfers (5D -> 2D), zodat je een plaatje kunt maken. Het kiest die 2 nieuwe cijfers zo, dat je zoveel mogelijk van de verschillen tussen de boeken blijft zien. Het is net als een foto van een 3D-voorwerp: je kiest de hoek waarop je het meeste ziet.

**Wat gebeurt er in de code hieronder?**
1. **PCA uitvoeren:** we vragen om 2 nieuwe assen, PC1 (links/rechts) en PC2 (omhoog/omlaag). PC1 laat de meeste verschillen tussen de boeken zien, PC2 de tweede meeste.
2. **Verklaarde variantie:** de code print hoeveel procent van alle verschillen elke as laat zien. Zijn die twee getallen samen hoog, dan is het plaatje betrouwbaar. Het stuk dat ontbreekt tot 100% is informatie die we kwijt zijn door van 5 naar 2 te gaan.
3. **Loadings:** een tabel die laat zien hoe PC1 en PC2 zijn opgebouwd uit de 5 oorspronkelijke kolommen. Een getal ver van 0 betekent dat die kolom zwaar meetelt voor die as. Een plus en een min betekenen dat de kolommen elkaars tegenpool zijn (de ene hoog, de andere laag). Zo kun je uitleggen wat een as eigenlijk betekent.
4. **Plaatje:** elk punt is een boek, gekleurd per niveau. Punten die dicht bij elkaar liggen zijn boeken die op elkaar lijken.

**Let op:** PCA is de enige van de drie technieken waarbij de assen echt iets betekenen. Bij t-SNE en UMAP zijn de assen alleen een plek op het plaatje.

In [ ]:
# --- PCA: lineaire reductie naar 2D ---
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Verklaarde variantie ratio (per component):", pca.explained_variance_ratio_)
print("Totaal verklaarde variantie:", pca.explained_variance_ratio_.sum())

# Loadings: welke features wegen zwaar mee in PC1 en PC2?
loadings = pd.DataFrame(pca.components_.T, index=features, columns=["PC1", "PC2"])
display(loadings.round(2))

plot_2d(X_pca, "PCA – 2D projectie", "PC1", "PC2")

##### t-SNE

**Wat doet t-SNE?**
t-SNE kijkt bij elk boek wie er in de buurt zit: welke boeken lijken het meest op dit boek? Daarna maakt het een plaatje waarin die buren ook echt naast elkaar liggen. Het is net als een klas die in groepjes gaat zitten: wie op elkaar lijkt, zit bij elkaar. Daardoor is t-SNE goed in het zichtbaar maken van groepjes (clusters).

**Wat gebeurt er in de code hieronder?**
1. **t-SNE uitvoeren:** we vragen om 2 nieuwe assen, zodat we een plaatje kunnen maken.
2. **Perplexity = 30:** dit getal bepaalt met hoeveel buren elk boek rekening houdt. Een laag getal kijkt naar heel kleine groepjes, een hoog getal naar grotere groepen. 30 is een gangbare beginwaarde.
3. **random_state = 42:** t-SNE gebruikt een beetje toeval. Met dit getal krijg je elke keer hetzelfde plaatje als je de cel opnieuw draait.
4. **Plaatje:** elk punt is een boek, gekleurd per niveau.

**Hoe lees je het plaatje?**
- Punten die dicht bij elkaar liggen, zijn boeken die op elkaar lijken. Dat mag je gebruiken.
- Een los groepje betekent dat die boeken op elkaar lijken en anders zijn dan de rest.
- De assen betekenen niets, het zijn alleen plekken op het plaatje. Ze hebben dus geen echte naam, daarom heten ze "Dim 1" en "Dim 2".
- De afstand tussen twee groepjes zegt niets. Een groepje dat ver weg ligt, is dus niet per se "anderder" dan een groepje dat dichtbij ligt.
- Er is geen "verklaarde variantie" zoals bij PCA, want t-SNE meet geen verschillen op één as.

In [ ]:
# --- t-SNE: focust op lokale structuur / clustervorming ---
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

plot_2d(X_tsne, "t-SNE – 2D projectie (perplexity=30)", "Dim 1", "Dim 2")

##### UMAP

**Wat doet UMAP?**
UMAP werkt een beetje zoals t-SNE: het kijkt bij elk boek wie er in de buurt zit en zet die buren in het plaatje bij elkaar. Het verschil is dat UMAP ook een beetje let op het grote geheel. Groepjes die in het echt dicht bij elkaar zitten, komen daardoor vaker ook op het plaatje dicht bij elkaar te liggen. UMAP is ook meestal sneller dan t-SNE.

**Wat gebeurt er in de code hieronder?**
1. **UMAP uitvoeren:** we vragen om 2 nieuwe assen, zodat we een plaatje kunnen maken.
2. **n_neighbors = 15:** met hoeveel buren houdt elk boek rekening? Een laag getal kijkt naar kleine groepjes, een hoog getal naar grotere groepen. 15 is een gangbare beginwaarde.
3. **min_dist = 0.1:** hoe dicht mogen punten in een groepje op elkaar zitten? Een laag getal geeft strakke, dichte groepjes. Een hoger getal spreidt de punten meer uit.
4. **random_state = 42:** UMAP gebruikt een beetje toeval. Met dit getal krijg je elke keer hetzelfde plaatje.
5. **Plaatje:** elk punt is een boek, gekleurd per niveau.

**Hoe lees je het plaatje?**
- Punten die dicht bij elkaar liggen, zijn boeken die op elkaar lijken.
- Een los groepje betekent dat die boeken op elkaar lijken en anders zijn dan de rest.
- De assen betekenen niets, net als bij t-SNE. Daarom heten ze "Dim 1" en "Dim 2".
- De afstand tussen groepjes zegt bij UMAP iets meer dan bij t-SNE, maar lees er niet te veel in.

**Eenmalig nodig:** staat UMAP nog niet op je computer? Verwijder het hekje voor `!pip install umap-learn`, draai die regel één keer en zet het hekje weer terug. Krijg je een waarschuwing (`UserWarning`) over `n_jobs`? Die is onschuldig: hij komt doordat we `random_state` hebben ingesteld.

In [ ]:
# --- UMAP: snel en behoudt lokale + deels globale structuur ---
import umap

umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

plot_2d(X_umap, "UMAP – 2D projectie (n_neighbors=15, min_dist=0.1)", "Dim 1", "Dim 2")

### Interacties

De interactiegegevens worden ingelezen, leespercentages worden genormaliseerd en de kwaliteit van de gegevens wordt samengevat.



In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dfInter = pd.read_csv("./dataset/InteractiesCSV.csv", sep=";")

In [ ]:
dfInter["leespercentage"] = dfInter["leespercentage"].str.removesuffix("%").astype(np.int64)

In [ ]:
dfInter

In [ ]:
dfInter.to_csv("./dataset/cleaned_Interacties.csv", index=False)

In [ ]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", dfInter.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": dfInter.nunique(),
    "Aantal missende waarden": dfInter.isna().sum(),
    "Datatype": dfInter.dtypes,
    "Aantal dubbele waarden": dfInter.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

### Student

De studentgegevens worden ingelezen, taalvoorkeuren worden gestandaardiseerd en de datakwaliteit wordt bekeken.



In [ ]:
dfStud = pd.read_csv("./dataset/StudentCSV.csv", sep=";")

In [ ]:
dfStud.head()

In [ ]:
dfStud.loc[dfStud["voorkeur_taal"] == "Nederlands", "voorkeur_taal"] = "NL"
dfStud.loc[dfStud["voorkeur_taal"] == "Engels", "voorkeur_taal"] = "EN"

In [ ]:
dfStud.to_csv("./dataset/cleaned_Student.csv", index=False)

In [ ]:
dfStud.head()

In [ ]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", dfStud.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": dfStud.nunique(),
    "Aantal missende waarden": dfStud.isna().sum(),
    "Datatype": dfStud.dtypes,
    "Aantal dubbele waarden": dfStud.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

## 2. NLP-tekstbewerking

In deze stap worden de catalogusteksten voorbereid voor tekstuele vergelijking.



### Leesmateriaal

De beschrijvende velden worden samengevoegd. Daarna worden tekst en leestekens opgeschoond, stopwoorden verwijderd en woorden met het Nederlandse spaCy-model gelemmatiseerd. De verwerkte tekst wordt aan de catalogus toegevoegd.



In [ ]:
#!pip install nltk

In [ ]:
#%pip install spacy

In [ ]:
#import sys
#!{sys.executable} -m spacy download nl_core_news_lg

In [ ]:
import pandas as pd
import nltk
import numpy as np

In [ ]:
df = pd.read_csv("./dataset/cleaned_Leesmateriaal.csv")
SamenKolom = df["Gecombineerd"]

In [ ]:
SamenKolom = SamenKolom.str.lower()

In [ ]:
#cijfers met spatie ervoor en erna verwijderen
SamenKolom = SamenKolom.str.replace(r'(?<=\s)\d+(?=\s)', '', regex=True)

In [ ]:
#puntkomma's en schuine strepen vervangen door spaties
SamenKolom = SamenKolom.str.replace(';', ' ')
SamenKolom = SamenKolom.str.replace('/', ' ')

In [ ]:
#alle overige leestekens verwijderen
SamenKolom = SamenKolom.str.replace(r'[^\w\s]', '', regex=True)

In [ ]:
#dubbele spaties verwijderen
SamenKolom = SamenKolom.str.replace(r'\s+', ' ', regex=True)
SamenKolom = SamenKolom.str.strip()

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('dutch'))

In [ ]:
#functie om stopwoorden te verwijderen
def verwijder_stopwoorden(zin):
    woorden = zin.split()
    nieuwe_woorden = []

    for woord in woorden:
        if woord.lower() not in stop_words:
            nieuwe_woorden.append(woord)

    return ' '.join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(verwijder_stopwoorden)

In [ ]:
#functie om woorden te stammen (SnowballStemmer voor Nederlands)
# from nltk.stem.snowball import SnowballStemmer

# stemmer = SnowballStemmer("dutch")

# def stam_tekst(zin):
#     woorden = zin.split()
#     gestamde_woorden = []

#     for woord in woorden:
#         gestamde_woorden.append(stemmer.stem(woord))

#     return " ".join(gestamde_woorden)

# SamenKolom = SamenKolom.apply(stam_tekst)

In [ ]:
#functie om woorden te lemmatiseren (Spacy voor Nederlands)
import spacy

nlp = spacy.load("nl_core_news_lg")

def lemmatiseer(zin):
    woorden = nlp(zin)
    nieuwe_woorden = []

    for woord in woorden:
        if not woord.is_punct:
            nieuwe_woorden.append(woord.lemma_)

    return " ".join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(lemmatiseer)

In [ ]:
df["nlp"] = SamenKolom

In [ ]:
df.to_csv("./dataset/cleaned_Leesmateriaal.csv", index=False)

### Interacties

Interactietype, leespercentage en beoordeling worden gecombineerd tot een interactiescore. Daarna wordt een student-itemmatrix opgebouwd voor de collaboratieve aanbevelingen.



In [ ]:
intdf = pd.read_csv("./dataset/cleaned_Interacties.csv")

In [ ]:

# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", intdf.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": intdf.nunique(),
    "Aantal missende waarden": intdf.isna().sum(),
    "Datatype": intdf.dtypes,
    "Aantal dubbele waarden": intdf.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

In [ ]:
intdf[1995:2000]

In [ ]:
intdf["interactietype"].unique().tolist()

In [ ]:
inttype = intdf["interactietype"]

In [ ]:
inttype.unique().tolist()

In [ ]:
#interactietypes waardes geven van 1 tot 5
inttype.loc[inttype == "geliket"] = "5"
inttype.loc[inttype == "opgeslagen"] = "4"
inttype.loc[inttype == "gelezen"] = "3"
inttype.loc[inttype == "bekeken"] = "2"
inttype.loc[inttype == "niet_afgemaakt"] = "1"
inttype.loc[inttype == "afgewezen"] = "1"
inttype = inttype.astype(float)

In [ ]:
leesprc = intdf["leespercentage"]

In [ ]:
#leespercentage waardes geven van 1 tot 5
leesprc = leesprc.apply(lambda x: round(x / 20))
leesprc = leesprc.astype(float)

In [ ]:
leesprc.unique().tolist()

In [ ]:
#interacties score berekenen, lege waardes worden niet meegenomen
scores = pd.DataFrame({
    "leesprc": leesprc,
    "inttype": inttype,
    "beoordeling": intdf["beoordeling"],
}, index=intdf.index)

weights = pd.Series({
    "leesprc": 0.15,
    "inttype": 0.35,
    "beoordeling": 0.50,
})

weighted_sum = scores.mul(weights, axis="columns").sum(axis=1, min_count=1)
available_weight = scores.notna().mul(weights, axis="columns").sum(axis=1)

intscore = weighted_sum / available_weight

In [ ]:
intscore

In [ ]:
intdf["intscore"] = intscore

In [ ]:
intdf.head()

In [ ]:
intdf.to_csv("./dataset/cleaned_Interacties.csv", index=False)

In [ ]:
intdf["intscore"] = intscore

student_item_matrix = intdf.pivot_table(
    index="student_id",
    columns="cataloguscode",
    values="intscore",
    aggfunc="mean"
)

student_item_matrix.head()

In [ ]:
student_item_matrix.to_csv("./matrix/student_item_matrix.csv", index=True)

## 3. Algoritme

De aanbevelingen worden op twee manieren opgebouwd: op basis van vergelijkbare studenten en op basis van de inhoud van leesmaterialen.



### Collaboratief

Collaborative filtering vergelijkt scorepatronen tussen studenten. De volgende cellen tonen vergelijkbare studenten, genereren aanbevelingen en evalueren de prestaties op testdata.



In [ ]:
# Vergelijk de scorepatronen van alle studenten op materialen.
# .T draait de matrix om, zodat corr() de studenten met elkaar vergelijkt.
# min_periods=3 vereist minstens 3 materialen met scores van beide studenten.
student_similarity = student_item_matrix.T.corr(min_periods=3)

# Maak een matrix waarin 1 betekent dat een score bekend is en 0 dat die ontbreekt.
# De matrixvermenigvuldiging telt per studentenpaar hoeveel materialen zij delen.
overlap = (
    student_item_matrix.notna().astype(int)
    @ student_item_matrix.notna().astype(int).T
)

# Kies voor dit voorbeeld de eerste student uit de matrix.
# De volgende cel gebruikt expliciet S001.
student_id = student_similarity.index[0]
# Pak de correlaties van deze student met alle andere studenten.
neighbors = (
    student_similarity[student_id]
    # Verwijder de student zelf: die is altijd volledig gelijk aan zichzelf.
    .drop(index=student_id)
    # Verwijder studenten waarvoor geen correlatie berekend kon worden.
    .dropna()
    # Zet de hoogste correlaties bovenaan.
    .sort_values(ascending=False)
    # Bewaar de 5 hoogste correlaties (die kunnen ook negatief zijn).
    .head(5)
)
# Toon de meest vergelijkbare studenten en het aantal gedeelde materialen
# Toon de gekozen student en de correlaties met diens vergelijkbare studenten.
print("Student:", student_id)
print("Vergelijkbare studenten:")
print(neighbors)
# Toon hoeveel materialen deze studenten met de gekozen student gemeen hebben.
print("Gedeelde materialen:")
print(overlap.loc[student_id, neighbors.index])

In [ ]:
# Kies de student voor wie we aanbevelingen willen maken.
target = "S001"

# Haal de correlaties van deze student met anderen op.
# Verwijder de student zelf en correlaties die niet berekend konden worden.
neighbors = student_similarity[target].drop(target).dropna()
# Houd alleen positief vergelijkbare studenten over en kies de 5 sterkste buren.
neighbors = neighbors[neighbors > 0].nlargest(5)

# Haal de scores op die deze buren aan alle materialen hebben gegeven.
neighbor_scores = student_item_matrix.loc[neighbors.index]

# Vermenigvuldig elke score met de gelijkenis van de buur en tel dit per materiaal op.
# min_count=1 zorgt dat een materiaal zonder scores NaN blijft in plaats van 0.
weighted_sum = neighbor_scores.mul(neighbors, axis=0).sum(axis=0, min_count=1)
# Tel per materiaal de gelijkenissen op van de buren die het hebben beoordeeld.
weight_sum = neighbor_scores.notna().mul(neighbors, axis=0).sum(axis=0)
# Deel de gewogen scoresom door de gebruikte gewichten voor een voorspelde score.
predicted_scores = weighted_sum / weight_sum

# Tel hoeveel buren voor elk materiaal een score hebben gegeven.
support = neighbor_scores.notna().sum(axis=0)
# Bepaal welke materialen de gekozen student nog niet heeft beoordeeld.
unseen = student_item_matrix.loc[target].isna()

# Houd alleen onbekende materialen over die door minstens 2 buren zijn beoordeeld.
# Sorteer op voorspelde score en selecteer de 10 hoogste aanbevelingen.
recommendations = (
    predicted_scores[unseen & (support >= 2)]
    .sort_values(ascending=False)
    .head(10)
)

# Toon de aanbevolen cataloguscodes met hun voorspelde scores.
print(recommendations)

In [ ]:
# Eén score per student-materiaalcombinatie; voorkom dat dezelfde combinatie
# zowel in train als test terechtkomt.
pairs = (
    intdf.groupby(["student_id", "cataloguscode"], as_index=False)["intscore"]
    .mean()
)

# Houd per student willekeurig 20% van de interacties achter.
test = pairs.groupby("student_id", group_keys=False).sample(
    frac=0.2, random_state=42
)
train = pairs.drop(index=test.index)

# Bouw de matrix opnieuw op met uitsluitend train-data.
students = pairs["student_id"].unique()
items = pairs["cataloguscode"].unique()

train_matrix = (
    train.pivot(index="student_id", columns="cataloguscode", values="intscore")
    .reindex(index=students, columns=items)
)

# Bereken opnieuw de student-similarity op train-data.
similarity = train_matrix.T.corr(min_periods=3)


def recommend(user_id, k=3):
    neighbors = similarity.loc[user_id].drop(index=user_id).dropna()
    neighbors = neighbors[neighbors > 0].nlargest(k)

    if neighbors.empty:
        return []

    neighbor_scores = train_matrix.loc[neighbors.index]
    weighted_sum = neighbor_scores.mul(neighbors, axis=0).sum(axis=0, min_count=1)
    weight_sum = neighbor_scores.notna().mul(neighbors, axis=0).sum(axis=0)
    predicted = weighted_sum / weight_sum

    support = neighbor_scores.notna().sum(axis=0)
    unseen = train_matrix.loc[user_id].isna()

    return predicted[unseen & (support >= 2)].nlargest(k).index.tolist()


# Definieer vooraf wat je als een relevante interactie ziet.
# 4.0 is een startpunt op jouw intscore-schaal; pas dit aan als dat beter
# aansluit bij jouw definitie van "interessant".
relevant_items = (
    test.loc[test["intscore"] >= 4.0]
    .groupby("student_id")["cataloguscode"]
    .apply(set)
)

results = []
k = 5

for user_id, relevant in relevant_items.items():
    recs = recommend(user_id, k=k)
    hits = [item in relevant for item in recs]

    precision_at_3 = sum(hits) / k

    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, is_relevant in enumerate(hits, start=1)
        if is_relevant
    )
    ideal_hits = min(k, len(relevant))
    idcg = sum(1 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    ndcg_at_3 = dcg / idcg if idcg else 0

    results.append({
        "student_id": user_id,
        "Precision@3": precision_at_3,
        "NDCG@3": ndcg_at_3,
    })

scores = pd.DataFrame(results)
print(scores[["Precision@3", "NDCG@3"]].mean())

In [ ]:
import pandas as pd
import numpy as np

In [ ]:

# Instellingen om te tweaken
# Aantal aanbevelingen; bepaalt ook de K in Precision@K en NDCG@K.
TOP_K = 3
# Maximumaantal meest vergelijkbare studenten dat voor voorspellingen wordt gebruikt.
N_NEIGHBORS = 5
# Minimumaantal gedeelde materialen waarop een studentcorrelatie wordt berekend.
MIN_COMMON_ITEMS = 2
# Minimumaantal gekozen buren dat een materiaal moet hebben beoordeeld.
MIN_ITEM_SUPPORT = 1
# Vanaf deze intscore telt een achtergehouden materiaal als relevant.
RELEVANCE_THRESHOLD = 4
# Deel van de interactieparen per student dat als testdata wordt achtergehouden.
TEST_FRACTION = 0.3
# Verschillende seeds voor herhaalde, reproduceerbare train/test-splits.
# Gebruik dezelfde lijst wanneer je verschillende instellingen vergelijkt.
RANDOM_STATES = [11, 21, 31, 41, 51, 61, 71, 81, 91, 1]
# True gebruikt alleen positief gecorreleerde buren; False laat ook andere correlaties toe.
ONLY_POSITIVE_NEIGHBORS = True
# Kies componenten voor de interactiescore door regels toe te voegen of weg te laten.
# De gewichten geven het relatieve belang; ze worden automatisch genormaliseerd.
SCORE_WEIGHTS = {
    # 'leesprc': 0.15,      # Afgeleid van leespercentage (omgezet naar een 0-5-schaal).
    # 'inttype': 0.25,      # Afgeleid van interactietype via eval_type_scores hieronder.
    'beoordeling': 0.75,  # De beoordeling uit de interactiedata.
}

# Lees de opgeschoonde data en bereken de interactiescore, onafhankelijk van de baselinecellen.
eval_raw = pd.read_csv('./dataset/cleaned_Interacties.csv')

eval_type_scores = {
    'geliket': 5.0,
    'opgeslagen': 4.0,
    'gelezen': 3.0,
    'bekeken': 2.0,
    'niet_afgemaakt': 1.0,
    'afgewezen': 1.0,
}
eval_unknown_types = (
    set(eval_raw['interactietype'].dropna().unique()) - set(eval_type_scores)
)
if eval_unknown_types:
    raise ValueError('Onbekende interactietypes: {}'.format(sorted(eval_unknown_types)))

eval_all_inputs = pd.DataFrame({
    'leesprc': (eval_raw['leespercentage'] / 20).round(),
    'inttype': eval_raw['interactietype'].map(eval_type_scores),
    'beoordeling': pd.to_numeric(eval_raw['beoordeling'], errors='coerce'),
})
if not SCORE_WEIGHTS:
    raise ValueError('SCORE_WEIGHTS moet minstens één scorecomponent bevatten.')
eval_unknown_components = set(SCORE_WEIGHTS) - set(eval_all_inputs.columns)
if eval_unknown_components:
    raise ValueError('Onbekende scorecomponenten: {}'.format(sorted(eval_unknown_components)))
if any(weight <= 0 for weight in SCORE_WEIGHTS.values()):
    raise ValueError('Alle gekozen scoregewichten moeten groter zijn dan 0.')
eval_inputs = eval_all_inputs[list(SCORE_WEIGHTS)]
eval_weights = pd.Series(SCORE_WEIGHTS)
eval_weighted_sum = eval_inputs.mul(eval_weights, axis='columns').sum(axis=1, min_count=1)
eval_available_weight = eval_inputs.notna().mul(eval_weights, axis='columns').sum(axis=1)
eval_raw['intscore'] = eval_weighted_sum / eval_available_weight

# Gemiddelde score per student-materiaalpaar; voorkom overlap tussen train en test.
eval_pairs = (
    eval_raw.groupby(['student_id', 'cataloguscode'], as_index=False)['intscore']
    .mean()
    .dropna(subset=['intscore'])
    .reset_index(drop=True)
)

# Voer voor elke seed de volledige split, aanbevelingen en evaluatie opnieuw uit.
def evaluate_one_split(eval_random_state):
    eval_rng = np.random.default_rng(eval_random_state)
    eval_test_indices = []
    for _, eval_user_pairs in eval_pairs.groupby('student_id', sort=False):
        if len(eval_user_pairs) < 2:
            continue
        eval_n_test = max(1, round(len(eval_user_pairs) * TEST_FRACTION))
        eval_n_test = min(eval_n_test, len(eval_user_pairs) - 1)
        eval_seed = int(eval_rng.integers(0, 2**32 - 1))
        eval_test_indices.extend(
            eval_user_pairs.sample(n=eval_n_test, random_state=eval_seed).index
        )

    if not eval_test_indices:
        raise ValueError('Niet genoeg student-materiaalparen voor de train/test-split.')

    eval_test = eval_pairs.loc[eval_test_indices].copy()
    eval_train = eval_pairs.drop(index=eval_test_indices).copy()
    eval_students = eval_pairs['student_id'].unique()
    eval_items = eval_pairs['cataloguscode'].unique()
    eval_train_matrix = (
        eval_train.pivot(index='student_id', columns='cataloguscode', values='intscore')
        .reindex(index=eval_students, columns=eval_items)
    )
    eval_similarity = eval_train_matrix.T.corr(min_periods=MIN_COMMON_ITEMS)

    def predict_scores_for_split(user_id):
        eval_neighbors = eval_similarity.loc[user_id].drop(index=user_id).dropna()
        if ONLY_POSITIVE_NEIGHBORS:
            eval_neighbors = eval_neighbors[eval_neighbors > 0]
        eval_neighbors = eval_neighbors.nlargest(N_NEIGHBORS)
        if eval_neighbors.empty:
            return pd.Series(dtype='float64')

        eval_neighbor_scores = eval_train_matrix.loc[eval_neighbors.index]
        eval_weighted_sum = eval_neighbor_scores.mul(
            eval_neighbors, axis=0
        ).sum(axis=0, min_count=1)
        eval_weight_sum = eval_neighbor_scores.notna().mul(
            eval_neighbors, axis=0
        ).sum(axis=0)
        eval_predicted = eval_weighted_sum / eval_weight_sum
        eval_support = eval_neighbor_scores.notna().sum(axis=0)
        eval_unseen = eval_train_matrix.loc[user_id].isna()
        eval_candidates = eval_unseen & (eval_support >= MIN_ITEM_SUPPORT)
        return eval_predicted[eval_candidates]

    eval_test_users = eval_test['student_id'].unique()
    eval_predicted_by_user = {
        eval_user_id: predict_scores_for_split(eval_user_id)
        for eval_user_id in eval_test_users
    }
    eval_recs_by_user = {
        eval_user_id: eval_scores.nlargest(TOP_K).index.tolist()
        for eval_user_id, eval_scores in eval_predicted_by_user.items()
    }
    eval_relevant_by_user = (
        eval_test.loc[eval_test['intscore'] >= RELEVANCE_THRESHOLD]
        .groupby('student_id')['cataloguscode']
        .apply(set)
        .to_dict()
    )
    eval_actual_scores = []
    eval_predicted_scores = []
    for eval_row in eval_test.itertuples(index=False):
        eval_prediction = eval_predicted_by_user[eval_row.student_id].get(
            eval_row.cataloguscode, np.nan
        )
        if pd.notna(eval_prediction):
            eval_actual_scores.append(eval_row.intscore)
            eval_predicted_scores.append(eval_prediction)
    if eval_predicted_scores:
        eval_errors = np.asarray(eval_actual_scores) - np.asarray(eval_predicted_scores)
        eval_mae = np.mean(np.abs(eval_errors))
        eval_rmse = np.sqrt(np.mean(np.square(eval_errors)))
        eval_train_mean = eval_train['intscore'].mean()
        eval_baseline_errors = np.asarray(eval_actual_scores) - eval_train_mean
        eval_baseline_mae = np.mean(np.abs(eval_baseline_errors))
        eval_baseline_rmse = np.sqrt(np.mean(np.square(eval_baseline_errors)))
    else:
        eval_mae = np.nan
        eval_rmse = np.nan
        eval_baseline_mae = np.nan
        eval_baseline_rmse = np.nan

    eval_precision_values = []
    eval_ndcg_values = []
    for eval_user_id, eval_relevant in eval_relevant_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_hits = [item in eval_relevant for item in eval_recs]
        eval_precision_values.append(sum(eval_hits) / TOP_K)
        eval_dcg = sum(
            1 / np.log2(rank + 1)
            for rank, is_relevant in enumerate(eval_hits, start=1)
            if is_relevant
        )
        eval_ideal_hits = min(TOP_K, len(eval_relevant))
        eval_idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, eval_ideal_hits + 1)
        )
        eval_ndcg_values.append(eval_dcg / eval_idcg if eval_idcg else 0.0)

    eval_known_items_by_user = (
        eval_test.groupby('student_id')['cataloguscode'].apply(set).to_dict()
    )
    eval_judged_precision_values = []
    eval_judged_shares = []
    eval_top_k_coverages = []
    eval_judged_user_count = 0
    for eval_user_id, eval_known_items in eval_known_items_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_judged_recs = [item for item in eval_recs if item in eval_known_items]
        eval_relevant = eval_relevant_by_user.get(eval_user_id, set())
        eval_judged_hits = sum(item in eval_relevant for item in eval_judged_recs)
        if eval_judged_recs:
            eval_judged_user_count += 1
            eval_judged_precision_values.append(
                eval_judged_hits / len(eval_judged_recs)
            )
        else:
            eval_judged_precision_values.append(np.nan)
        eval_judged_shares.append(
            len(eval_judged_recs) / len(eval_recs) if eval_recs else np.nan
        )
        eval_top_k_coverages.append(len(eval_judged_recs) / TOP_K)

    eval_total_pairs = len(eval_pairs)
    return {
        'random_state': eval_random_state,
        'train_percent': 100 * len(eval_train) / eval_total_pairs,
        'test_percent': 100 * len(eval_test) / eval_total_pairs,
        'students_with_relevant_test': len(eval_precision_values),
        'students_with_judged_recommendations': eval_judged_user_count,
        'average_recommendations_returned': pd.Series(
            [len(recs) for recs in eval_recs_by_user.values()], dtype='float64'
        ).mean(),
        'Precision@K': pd.Series(eval_precision_values, dtype='float64').mean(),
        'NDCG@K': pd.Series(eval_ndcg_values, dtype='float64').mean(),
        'Judged-only Precision@K': pd.Series(
            eval_judged_precision_values, dtype='float64'
        ).mean(),
        'judged_share_of_recommendations': pd.Series(
            eval_judged_shares, dtype='float64'
        ).mean(),
        'judged_top_k_coverage': pd.Series(
            eval_top_k_coverages, dtype='float64'
        ).mean(),
        'MAE': eval_mae,
        'RMSE': eval_rmse,
        'Global-mean baseline MAE': eval_baseline_mae,
        'Global-mean baseline RMSE': eval_baseline_rmse,
        'score_prediction_coverage': len(eval_predicted_scores) / len(eval_test),
        'predicted_test_pairs': len(eval_predicted_scores),
    }

if not RANDOM_STATES:
    raise ValueError('RANDOM_STATES moet minstens ??n seed bevatten.')
if len(set(RANDOM_STATES)) != len(RANDOM_STATES):
    raise ValueError('RANDOM_STATES mag geen dubbele seeds bevatten.')

eval_run_metrics = pd.DataFrame(
    [evaluate_one_split(eval_seed) for eval_seed in RANDOM_STATES]
)

eval_component_sources = {
    'leesprc': 'leespercentage / 20, afgerond',
    'inttype': 'interactietype (gescoord via eval_type_scores)',
    'beoordeling': 'beoordeling',
}
print('Gebruikte scorecomponenten en ingestelde relatieve gewichten:')
for eval_component, eval_weight in SCORE_WEIGHTS.items():
    print('  {} <- {}: {}'.format(
        eval_component, eval_component_sources[eval_component], eval_weight
    ))
print('Instellingen (1/2): splits={} | test={:.0f}% | train/test={:.1f}/{:.1f}% | K={}'.format(
    len(RANDOM_STATES), 100 * TEST_FRACTION,
    eval_run_metrics['train_percent'].mean(), eval_run_metrics['test_percent'].mean(),
    TOP_K
))
print('Instellingen (2/2): buren={} | overlap={} | support={} | drempel={} | seeds={}'.format(
    N_NEIGHBORS, MIN_COMMON_ITEMS, MIN_ITEM_SUPPORT, RELEVANCE_THRESHOLD, RANDOM_STATES
))
print('Gemiddelde +/- standaardafwijking tussen splits:')

def print_run_summary(eval_label, eval_column):
    eval_values = eval_run_metrics[eval_column].dropna()
    if eval_values.empty:
        print('{}: geen geldige resultaten'.format(eval_label))
        return
    print('{}: {:.4f} +/- {:.4f}'.format(
        eval_label, eval_values.mean(), eval_values.std(ddof=0)
    ))

for eval_label, eval_column in [
    ('Precision@{}'.format(TOP_K), 'Precision@K'),
    ('NDCG@{}'.format(TOP_K), 'NDCG@K'),
    ('Judged-only Precision@{}'.format(TOP_K), 'Judged-only Precision@K'),
    ('Aandeel beoordeeld in aanbevelingen', 'judged_share_of_recommendations'),
    ('Studenten met relevante testitems', 'students_with_relevant_test'),
    ('Studenten met beoordeelde aanbevelingen', 'students_with_judged_recommendations'),
]:
    print_run_summary(eval_label, eval_column)


In [ ]:
# Voer eerst cel 2 uit; die berekent MAE/RMSE per random split.
# MAE is makkelijk te interpreteren; RMSE bestraft grote fouten extra zwaar.
print('Scorevoorspelling op testparen (gemiddelde +/- SD over random splits):')
print_run_summary('MAE', 'MAE')
print_run_summary('RMSE', 'RMSE')
print_run_summary('Voorspellingsdekking (fractie testparen)', 'score_prediction_coverage')

In [ ]:
# Run cel 2 eerst. De baseline voorspelt voor elk testpaar het gemiddelde van de train-scores.
# De vergelijking gebruikt alleen testparen waarvoor collaborative filtering een score kon voorspellen.
eval_comparison = pd.DataFrame({
    'Model': ['Collaborative filtering', 'Gemiddelde train-score baseline'],
    'MAE gemiddeld': [
        eval_run_metrics['MAE'].mean(),
        eval_run_metrics['Global-mean baseline MAE'].mean(),
    ],
    'MAE SD': [
        eval_run_metrics['MAE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline MAE'].std(ddof=0),
    ],
    'RMSE gemiddeld': [
        eval_run_metrics['RMSE'].mean(),
        eval_run_metrics['Global-mean baseline RMSE'].mean(),
    ],
    'RMSE SD': [
        eval_run_metrics['RMSE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline RMSE'].std(ddof=0),
    ],
})
print('Fout op dezelfde voorspelbare testparen (lager is beter):')
print(eval_comparison.round(4).to_string(index=False))
print('Dekking collaborative filtering: {:.1f}%'.format(
    100 * eval_run_metrics['score_prediction_coverage'].mean()
))

### Content-based

De content-based recommender maakt TF-IDF-vectoren van de leesmateriaalteksten en vergelijkt die met de interesses en voorkeuren van een student. Ook kunnen materialen onderling op tekstuele overeenkomst worden vergeleken.



In [ ]:
#TF-IDF vectorizer importeren
from sklearn.feature_extraction.text import TfidfVectorizer

teksten = SamenKolom.fillna("").astype(str)

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(teksten)

print(X_tfidf)

In [ ]:
#TF-IDF resultaten omzetten naar DataFrame
import pandas as pd

tfidf_df = pd.DataFrame.sparse.from_spmatrix(
    X_tfidf,
    index=SamenKolom.index,
    columns=vectorizer.get_feature_names_out()
)

tfidf_df.head()

In [ ]:
SamenKolom[0:3]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Vergelijk alle leesmaterialen op basis van hun TF-IDF-vectoren.
cosine_sim = cosine_similarity(X_tfidf)
cataloguscodes = df["cataloguscode"].astype(str)
cosine_sim_df = pd.DataFrame(
    cosine_sim,
    index=cataloguscodes,
    columns=cataloguscodes
)

# Bekijk een klein deel van de matrix.
display(cosine_sim_df.iloc[:5, :5])

In [ ]:
student_profiles = pd.read_csv("./dataset/cleaned_Student.csv").set_index("student_id")
materiaal_df = df.set_index("cataloguscode")

def aanbevolen_leesmateriaal(student_id, aantal=5):
    """Beveel beschikbare materialen aan die passen bij het profiel van een student."""
    student_id = str(student_id)
    if student_id not in student_profiles.index:
        raise KeyError(f"Onbekend student_id: {student_id}")
    if aantal < 1:
        raise ValueError("aantal moet minstens 1 zijn")

    profiel = student_profiles.loc[student_id]
    max_leestijd = pd.to_numeric(profiel["max_leestijd_min"], errors="coerce")
    if pd.isna(max_leestijd):
        raise ValueError(f"Geen geldige maximale leestijd voor student {student_id}")

    taal = str(profiel["voorkeur_taal"]).strip().casefold()
    beschikbare_boeken = (
        df["taal"].fillna("").astype(str).str.strip().str.casefold().eq(taal)
        & df["gegevensstatus"].fillna("").astype(str).str.strip().str.casefold().eq("beschikbaar")
        & pd.to_numeric(df["geschatte_leestijd_min"], errors="coerce").le(max_leestijd)
    )
    posities = beschikbare_boeken.to_numpy().nonzero()[0]
    if len(posities) == 0:
        raise ValueError(
            f"Geen beschikbare materialen gevonden voor taal {taal!r} "
            f"met een leestijd van maximaal {max_leestijd:g} minuten"
        )

    profiel_onderdelen = [
        profiel["interesses"],
        profiel["voorkeur_genre"],
        profiel["leesniveau"],
    ]
    profiel_tekst = " ".join(
        str(onderdeel).strip()
        for onderdeel in profiel_onderdelen
        if pd.notna(onderdeel)
        and str(onderdeel).strip().casefold() not in {"", "geen voorkeur"}
    )
    profiel_vector = vectorizer.transform([profiel_tekst])
    if profiel_vector.nnz == 0:
        raise ValueError(
            f"De profieltermen van student {student_id} komen niet voor in de boekteksten"
        )

    scores = cosine_similarity(profiel_vector, X_tfidf).ravel()[posities]
    resultaten = df.iloc[posities].copy()
    resultaten["profiel_similarity"] = scores

    niveau = str(profiel["leesniveau"]).strip().casefold()
    genre = str(profiel["voorkeur_genre"]).strip().casefold()
    materiaaltype = str(profiel["voorkeur_materiaaltype"]).strip().casefold()
    resultaten["zelfde_leesniveau"] = (
        resultaten["Niveau"].fillna("").astype(str).str.strip().str.casefold().eq(niveau)
    )
    resultaten["genre_match"] = (
        resultaten["genre"].fillna("").astype(str).str.strip().str.casefold().eq(genre)
        if genre not in {"", "geen voorkeur"}
        else False
    )
    resultaten["materiaaltype_match"] = (
        resultaten["Type materiaal"].fillna("").astype(str).str.strip().str.casefold().eq(materiaaltype)
        if materiaaltype not in {"", "gemengd", "geen voorkeur"}
        else False
    )

    resultaten = resultaten.sort_values(
        ["profiel_similarity", "zelfde_leesniveau", "genre_match", "materiaaltype_match"],
        ascending=False,
    )
    return resultaten[
        [
            "cataloguscode",
            "Titel",
            "Niveau",
            "genre",
            "Type materiaal",
            "geschatte_leestijd_min",
            "profiel_similarity",
        ]
    ].head(aantal)


def vergelijkbare_leesmaterialen(cataloguscode, aantal=5):
    """Geef de meest vergelijkbare materialen voor een cataloguscode."""
    cataloguscode = str(cataloguscode)
    if cataloguscode not in cosine_sim_df.index:
        raise KeyError(f"Onbekende cataloguscode: {cataloguscode}")
    if aantal < 1:
        raise ValueError("aantal moet minstens 1 zijn")

    scores = cosine_sim_df.loc[cataloguscode].drop(cataloguscode).nlargest(aantal)
    resultaten = materiaal_df.loc[scores.index, ["Titel"]].copy()
    resultaten["cosine_similarity"] = scores.values
    return resultaten

# Voorbeelden: profielaanbevelingen en vergelijkbare materialen.
display(aanbevolen_leesmateriaal("S001", aantal=5))
display(vergelijkbare_leesmaterialen("CAT-00001", aantal=5))